# Verify Saved Results
Replay the samples generated from the fixed random seeds, recompute the fitting and queries for the first repetition, and verify that the aggregated results are consistent with the raw records.

In [1]:
import json, sys, types
from pathlib import Path

def _load_core_notebook():
    notebook_path = Path("00_bn_core.ipynb")
    document = json.loads(notebook_path.read_text(encoding="utf-8"))
    module = types.ModuleType("bn_core")
    module.__file__ = str(notebook_path.resolve())
    sys.modules["bn_core"] = module
    for cell_index, cell in enumerate(document["cells"]):
        if cell.get("cell_type") == "code":
            source = "".join(cell.get("source", []))
            exec(compile(source, f"{notebook_path}#cell-{cell_index}", "exec"), module.__dict__)
    return module

bn_core = _load_core_notebook()


In [2]:
"""Replay the saved first-repetition samples and independently re-evaluate
all saved first-repetition graph/sample-size fits, including exact queries.
Also checks that summary entries agree with their raw CSV sources.
"""
from pathlib import Path
import csv
import hashlib
import json
import math
import numpy as np
from bn_core import *

ROOT = Path.cwd().resolve()


def load_csv(name):
    with (ROOT/"results"/name).open(encoding="utf-8") as f:
        return list(csv.DictReader(f))


def main():
    config=json.loads((ROOT/"config.json").read_text())
    environment=json.loads((ROOT/"results"/"environment.json").read_text())
    summary=json.loads((ROOT/"results"/"summary.json").read_text())
    checks=0
    for ni,(name,spec) in enumerate(config["networks"].items()):
        path=ROOT/"data"/(name+".bif.gz")
        assert hashlib.sha256(path.read_bytes()).hexdigest()==environment["data"][name]["sha256"]
        bn=read_bif(path);cards=bn.cards;p=len(cards)
        evidence={bn.names.index(v):bn.states[bn.names.index(v)].index(x) for v,x in spec["evidence"].items()}
        R=set(range(p))-set(evidence)
        orders=json.loads((ROOT/"data"/f"{name}_orders.json").read_text())
        rng=np.random.default_rng(np.random.SeedSequence([config["seed"],ni,0]))
        for order in orders:
            beta=topological_order(p,bn.edges,rng)
            assert [bn.names[v] for v in beta]==order["beta"]
        samples=np.load(ROOT/"data"/f"{name}_conditional_samples.npz")
        X=samples["replicate_00"]
        rng=np.random.default_rng(np.random.SeedSequence([config["seed"],ni,1,0]))
        regenerated,proposals=sample_conditional(bn,len(X),evidence,rng,summary[name]["evidence_probability"])
        assert np.array_equal(X,regenerated),"Seed does not regenerate saved sample"
        assert proposals==int(load_csv(f"{name}_sampling.csv")[0]["proposals"])
        train=X[config["pilot_size"]:config["pilot_size"]+max(config["sample_sizes"])]
        parents={}
        for order in orders:
            gid=order["graph_id"]
            if gid not in parents:
                beta=[bn.names.index(v) for v in order["beta"]]
                E,_,_=fgcr(p,bn.edges,set(evidence),beta)
                parents[gid]=graph_parents(R,E)
        parents[-1]=graph_parents(R,{(u,v) for u,v in bn.edges if u in R and v in R})
        cache={}
        query=tuple(bn.names.index(v) for v in spec["query"])
        event=tuple(bn.states[v].index(x) for v,x in zip(query,spec["event"]))
        raw=load_csv(f"{name}_raw_fits.csv")
        for row in raw:
            if int(row["replicate"])!=0:continue
            gid,n=int(row["graph_id"]),int(row["n"])
            fit=fit_tables(train[:n],parents[gid],cards,config["iss"])
            truth={}
            for v,pa in parents[gid].items():
                if (v,pa) not in cache:cache[v,pa]=true_family(bn,evidence,v,pa)
                truth[v]=cache[v,pa]
            actual=max(0.,cross_entropy(fit,truth)-summary[name]["conditional_entropy"])
            assert abs(actual-float(row["kl"]))<1e-11
            q=marginal(fitted_factors(fit,cards),query,cards)
            assert abs(float(q[event])-float(row["query_estimate"]))<1e-12
            if gid>=0:
                bs=weighted_bounds(fit,truth,n,config["delta"])
                for key in ["oracle_bound","empirical_bound"]:
                    assert abs(bs[key]-float(row[key]))<1e-9
            checks+=1
        selected=load_csv(f"{name}_selection.csv")
        for aggregate in summary[name]["selection"]:
            rs=[r for r in selected if int(r["n"])==aggregate["n"] and r["method"]==aggregate["method"]]
            assert len(rs)==config["replicates"]
            assert abs(float(np.mean([float(r["kl"]) for r in rs]))-aggregate["mean_kl"])<1e-12
            assert abs(math.sqrt(np.mean([float(r["query_error"])**2 for r in rs]))-aggregate["query_rmse"])<1e-12
    result=dict(status="passed",seed_replay_networks=2,raw_fit_recomputations=checks,
                summary_raw_consistency="passed",source_hashes="passed")
    (ROOT/"results"/"replay_verification.json").write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))


In [3]:
main()


{
  "status": "passed",
  "seed_replay_networks": 2,
  "raw_fit_recomputations": 171,
  "summary_raw_consistency": "passed",
  "source_hashes": "passed"
}
